# Braindecode + MOABB EEG 实验

按顺序运行各单元。使用本地 BCI Competition IV 2a 数据；每名受试者独立使用 T 会话的 288 个 Trial 训练、E 会话的 288 个 Trial 测试。取消训练内部划分。

## 1. 导入库


In [1]:
from pathlib import Path
from datetime import datetime
import json

import matplotlib.pyplot as plt
import numpy as np
import torch
from torch import nn
from sklearn.metrics import accuracy_score, balanced_accuracy_score, cohen_kappa_score, confusion_matrix, f1_score
from sklearn.model_selection import LeaveOneGroupOut
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import FunctionTransformer, StandardScaler
from sklearn.base import BaseEstimator, TransformerMixin

import braindecode
import moabb
from braindecode import EEGClassifier
from braindecode.models import EEGNet as BDEEGNet, Deep4Net, ShallowFBCSPNet,ATCNet, TCFormer
from braindecode.util import set_random_seeds
from moabb.evaluations import CrossSessionEvaluation
from moabb.paradigms import MotorImagery

import eegnet_model
from data_adapter import adapt_input, check_local_data, load_trial_info, make_dataset

c:\Users\hp\.conda\envs\moabb\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
c:\Users\hp\.conda\envs\moabb\Lib\site-packages\braindecode\models\eegpt.py:497: FutureWarning: Montage name 'standard_1020' is deprecated and will be removed in MNE 1.14. Use 'colin27_1020' instead.
  montage = mne.channels.make_standard_montage("standard_1020")
c:\Users\hp\.conda\envs\moabb\Lib\site-packages\braindecode\models\eegpt.py:1452: FutureWarning: Montage name 'standard_1020' is deprecated and will be removed in MNE 1.14. Use 'colin27_1020' instead.
  montage = make_standard_montage("standard_1020")


## 2. 数据读取与结果目录配置


In [2]:

PROJECT_ROOT = Path.cwd().resolve()
DATA_CONFIG = {
    "data_root": PROJECT_ROOT / "Data",
    "subjects": list(range(1, 10)),
}

RESULTS_ROOT = PROJECT_ROOT / "results"
# 选择结果的保存名称
OUTPUT_MODE = "model"  # "model" 或 "experiment"
EXPERIMENT_NAME = "EEGNet_750epoch_lr1e-2"
SAVE_CHECKPOINT = False


TRAIN_CONFIG = {
    "epochs": 750,
    "batch_size": 64,
    "learning_rate": 0.001,
    "weight_decay": 1e-4,
    "optimizer": "Adam",
    "loss": "CrossEntropyLoss",
    "seed": 2026,
    "device": "cuda",       # 手动选择 "cuda" 或 "cpu"
    "evaluation": "train_T_test_E",  # 固定单向 T→E，每人训练一次
    "validation_split": None,
}

# TCFormer专属
# TRAIN_CONFIG = {
#     "epochs": 750,
#     "batch_size": 64,
#     "learning_rate": 9e-4,
#     "weight_decay": 1e-3,
#     "optimizer": "Adam",
#     "loss": "CrossEntropyLoss",
#     "seed": 2026,
#     "device": "cuda",       # 手动选择 "cuda" 或 "cpu"
#     "evaluation": "train_T_test_E",  # 固定单向 T→E，每人训练一次
#     "validation_split": None,
# }

## 3. 模型结构参数与单模型选择


In [3]:
# 每个模型使用独立参数字典；每次只选择一个模型
MODEL_CONFIGS = {
    "CustomEEGNet": {
        "dropoutRate": 0.5, "kernelLength": 32, "kernelLength2": 16,
        "F1": 16, "D": 2, "F2": 32,
    },
    "CustomEEGTCNet": {
        "F1": 32, "D": 2, "kernelLength": 32, "poolSize": 8,
        "kernelLength2": 4, "dropoutRate": 0.5, "dropoutRate2": 0.5,
        "dropout_atten": 0.3, "tcn_filters": 64, "tcn_kernelSize": 4,
        "tcn_dropout": 0.3,
    },
    "EEGNex": {
        "F1": 8, "F2": 32, "D": 2
    },
    "BDEEGNet": {},
    # "BDEEGNet": {
    # "F1": 16,
    # "D": 2,
    # "F2": 32,
    # "kernel_length": 32,
    # "depthwise_kernel_length": 16,
    # "pool1_kernel_size": 8,
    # "pool2_kernel_size": 16,
    # "drop_prob": 0.5,
    # "pool_mode": "mean",
    # "batch_norm_momentum": 0.1,
    # "batch_norm_eps": 1e-5,
    # },
    "ATCNet": {},
    "TCFormer":{
        "n_transformer_layers": 2,
    },
    "Deep4Net": {},
    "ShallowFBCSPNet": {},
}
MODEL_NAME = "CustomEEGTCNet"

## 4. 模型构造与训练前检查


In [ ]:
def build_model(name, n_chans, n_times, n_classes):
    params = dict(MODEL_CONFIGS[name])
    if name == "CustomEEGNet":
        model = eegnet_model.EEGNet(
            channels=n_chans, 
            samples=n_times, 
            n_classes=n_classes, 
            **params
        )
    elif name == "CustomEEGTCNet":
        model = eegnet_model.EEGTCNet(
            channels=n_chans,
            n_classes=n_classes, 
            **params
            )
    elif name == "EEGNex":
        model = eegnet_model.EEGNex(
            channels=n_chans, 
            samples=n_times,
            n_classes=n_classes, 
            **params
        )
    else:
        model_class = {
            "TCFormer": TCFormer,
            "ATCNet": ATCNet,
            "BDEEGNet": BDEEGNet,
            "Deep4Net": Deep4Net,
            "ShallowFBCSPNet": ShallowFBCSPNet,
        }[name]
        model = model_class(
            n_chans=n_chans, 
            n_times=n_times, 
            n_outputs=n_classes, 
            **params
        )
    return model


def check_model(model, name, n_chans, n_times, n_classes):
    shape = (32, 1, n_chans, n_times)
    sample = torch.randn(shape)
    try:
        model.eval()
        output = model(torch.from_numpy(adapt_input(sample[:, 0].numpy(), name)))
        if tuple(output.shape) != (32, n_classes):
            raise ValueError(f"模型输出形状 {tuple(output.shape)}，预期 (32,{n_classes})")
        if not torch.isfinite(output).all():
            raise ValueError("模型输出含非有限数值")
        loss = nn.CrossEntropyLoss()(output, torch.randint(0, n_classes, (output.shape[0],), device=output.device))
        loss.backward()
    except Exception as exc:
        raise ValueError(f"{name} 与当前 Trial 形状或 CrossEntropyLoss 不兼容: {exc}") from exc
    print(f"模型输入: {shape} -> 实际输入: {tuple(adapt_input(sample[:, 0].numpy(), name).shape)}")
    print(f"模型输出: {tuple(output.shape)}；可训练参数: {sum(p.numel() for p in model.parameters() if p.requires_grad):,}")
    print(f"CrossEntropyLoss 前向/反向检查通过: {loss.item():.4f}")
    print("\n")

## 5. 逐折记录


In [ ]:
from skorch.callbacks import Callback

"""复现 EEG-TCNet 作者对数据标准化的代码：每通道、每时间点在训练 Trial 上拟合。"""
class ChannelStandardizer(BaseEstimator, TransformerMixin):

    def fit(self, X, y=None):
        X = np.asarray(X)
        if X.ndim != 3:
            raise ValueError(f"预期 (Trial, 通道, 时间点)，实际 {X.shape}")

        self.n_channels_, self.n_times_ = X.shape[1:]
        self.scalers_ = [
            StandardScaler(with_mean=True, with_std=True).fit(X[:, ch, :])
            for ch in range(self.n_channels_)
        ]
        return self

    def transform(self, X):
        X = np.asarray(X)
        if X.ndim != 3 or X.shape[1:] != (self.n_channels_, self.n_times_):
            raise ValueError(f"标准化输入维度不匹配：{X.shape}")

        result = np.empty(X.shape, dtype=np.float32)
        for ch, scaler in enumerate(self.scalers_):
            result[:, ch, :] = scaler.transform(X[:, ch, :])
        return result


"""在内存中保存 Train Loss 最低的模型，训练结束后自动恢复。"""
class BestTrainLoss(Callback):
  
    def on_train_begin(self, net, **kwargs):
        # 每名受试者开始训练时，重置最佳记录
        self.best_loss = float("inf")
        self.best_epoch = 0
        self.best_state = None

    def on_epoch_end(self, net, **kwargs):
        # 获取当前 Epoch 的平均训练 Loss
        current_loss = float(net.history[-1, "train_loss"])

        # 当前 Loss 更低，则更新最佳模型
        if np.isfinite(current_loss) and current_loss < self.best_loss:
            self.best_loss = current_loss
            self.best_epoch = len(net.history)

            # 复制模型权重到 CPU 内存，不写入磁盘
            self.best_state = {
                key: value.detach().cpu().clone()
                for key, value in net.module_.state_dict().items()
            }

    def on_train_end(self, net, **kwargs):
        # Epoch 结束后，恢复最佳权重
        if self.best_state is None:
            raise RuntimeError("训练期间未获得有效的最佳模型")

        net.module_.load_state_dict(self.best_state)

        # 提供给 FoldScorer 记录
        net.best_epoch_ = self.best_epoch
        net.best_train_loss_ = self.best_loss

        # 释放额外保存的权重副本
        self.best_state = None

        print(
            f"已恢复最佳模型：Epoch {self.best_epoch}，"
            f"Train Loss = {self.best_loss:.6f}"
        )


# EEGClassifier 是 Braindecode 提供的 EEG 神经网络分类器，它将 PyTorch 神经网络包装成类似 scikit-learn 的分类模型。
# 不必每次手动编写完整的 PyTorch 训练循环，就能够训练、验证和使用 EEG 深度学习模型。
class ReportingEEGClassifier(EEGClassifier):
    """保留本折训练样本，用于训练折混淆矩阵。"""
    # 调用官方分类器训练模型，并保存本次训练使用的输入和标签 在官方训练器的基础上增加了训练数据记录功能。
    def fit(self, X, y=None, **fit_params):
        fitted = super().fit(X, y, **fit_params)
        self.fit_X_ = X
        self.fit_y_ = np.asarray(y)
        return fitted

# 使用训练完成的模型评估测试集，计算准确率、Loss 和混淆矩阵所需的数据 某名受试者训练完成后，对测试集进行预测，记录结果，并返回准确率。
class FoldScorer:
    """MOABB 评分时收集Train中最好的一轮模型预测、Loss 和训练曲线。"""

    def __init__(self, n_classes, save_checkpoint):
        self.n_classes = n_classes
        self.save_checkpoint = save_checkpoint
        self.records = []

    # estimator 是已经训练完成的Pipeline模型，X、y 是当前要评分的数据与真实标签。
    def __call__(self, estimator, X, y):

        X_model = estimator[:-1].transform(X)  # 对测试数据使用训练数据已经标准化好的参数进行标准化 + 调整输入维度
        clf = estimator.named_steps["classifier"] # 从 scikit-learn 的 Pipeline 中，取出名称为 "classifier" 的步骤所对应的分类器对象，并赋值给 clf。
        logits = clf.forward(X_model, training=False)
        pred = logits.argmax(dim=1).detach().cpu().numpy()

        train_pred = clf.predict(clf.fit_X_)

        test_loss = nn.CrossEntropyLoss()(
            logits,
            torch.as_tensor(y, dtype=torch.long, device=logits.device),
        ).item()
        
        
        record = {
            "n_train": len(clf.fit_y_),
            "train_loss": [float(row["train_loss"]) for row in clf.history],
            "train_matrix": confusion_matrix(
                clf.fit_y_, train_pred, labels=range(self.n_classes)
            ),

            "n_test": len(y),
            "test_true": np.asarray(y),
            "test_pred": np.asarray(pred),
            "test_loss": float(test_loss),

            "best_epoch": int(clf.best_epoch_),
            "best_train_loss": float(clf.best_train_loss_),
        }
        if self.save_checkpoint:
            record["state_dict"] = {
                key: value.detach().cpu().clone()
                for key, value in clf.module_.state_dict().items()
            }
        self.records.append(record)
        return accuracy_score(y, pred)

    def fold_id(self, estimator, X, y):
        return float(len(self.records) - 1)


class TTrainETestSplit(LeaveOneGroupOut):
    """MOABB 按受试者调用：只生成 0train(T) → 1test(E) 的一次划分。"""

    def __init__(self):
        pass

    def get_n_splits(self, X=None, y=None, groups=None):
        return 1

    def split(self, X, y=None, groups=None):
        sessions = np.asarray(groups)
        if set(np.unique(sessions)) != {"0train", "1test"}:
            raise ValueError(f"预期 T/E 两个会话，实际为 {np.unique(sessions)}")
        
        train = np.flatnonzero(sessions == "0train")
        test = np.flatnonzero(sessions == "1test")
        
        if len(train) != 288 or len(test) != 288:
            raise ValueError(
                f"每个会话应有 288 个 Trial，实际 T={len(train)}, E={len(test)}；"
                "请检查数据完整性及预处理配置"
            )
        print(f"T→E 划分：训练 {len(train)}，测试 {len(test)}，验证 0")
        yield train, test

## 6. 图表与结果报告


In [6]:
def output_dir():
    if OUTPUT_MODE == "model":
        original_names = {"CustomEEGNet": "EEGNet", "CustomEEGTCNet": "EEGTCNet"}
        return RESULTS_ROOT / original_names.get(MODEL_NAME, MODEL_NAME)
    if OUTPUT_MODE == "experiment":
        return RESULTS_ROOT / EXPERIMENT_NAME
    raise ValueError("OUTPUT_MODE 只能是 model 或 experiment")


def save_curve(path, train_loss):
    path.parent.mkdir(parents=True, exist_ok=True)
    plt.figure(figsize=(8, 4))
    plt.plot(range(1, len(train_loss) + 1), train_loss, label="Train Loss")
    plt.xlabel("Epoch")
    plt.ylabel("Loss")
    plt.legend()
    plt.tight_layout()
    plt.savefig(path)
    plt.close()


def save_matrix(path, matrix, title, labels):
    path.parent.mkdir(parents=True, exist_ok=True)
    fig, ax = plt.subplots(figsize=(6, 5))
    image = ax.imshow(matrix, cmap="Blues")
    for i in range(len(labels)):
        for j in range(len(labels)):
            ax.text(j, i, str(matrix[i, j]), ha="center", va="center")
    ax.set(xticks=range(len(labels)), yticks=range(len(labels)),
           xticklabels=labels, yticklabels=labels,
           xlabel="Predicted", ylabel="True", title=title)
    fig.colorbar(image, ax=ax)
    fig.tight_layout()
    fig.savefig(path)
    plt.close(fig)


"""
指标	含义
Accuracy	所有 Trial 中预测正确的比例
Macro-F1	各类 F1 的算术平均
Kappa	考虑随机一致性的分类性能
Balanced Accuracy	各类别召回率的平均
Loss	测试集交叉熵
"""
def metrics_for(y_true, y_pred, loss):
    return {
        "accuracy": accuracy_score(y_true, y_pred),
        "macro_f1": f1_score(
            y_true, y_pred, 
            average="macro", 
            zero_division=0),
        "kappa": cohen_kappa_score(y_true, y_pred),
        "balanced_accuracy": balanced_accuracy_score(y_true, y_pred),
        "loss": loss,
    }


def write_results(folder, rows, recorder, classes, preview):
    folder.mkdir(parents=True, exist_ok=True)
    subject_records = {}
    fold_lines = []
    for row in rows.itertuples(index=False):
        fold_id = int(row.score_fold_id)
        record = recorder.records[fold_id]
        subject = int(row.subject)
        session = str(row.session)
        subject_records.setdefault(subject, []).append(record)
        subdir = folder / f"sub{subject}"
        save_curve(subdir / f"loss_curve_fold{fold_id}.png",
                   record["train_loss"])
        fold_metric = metrics_for(
            record["test_true"], record["test_pred"], record["test_loss"]
        )
        fold_lines.append((subject, session, fold_id, fold_metric))
        if SAVE_CHECKPOINT:
            torch.save(record["state_dict"], subdir / f"model_last_fold{fold_id}.pt")

    subject_metrics = {}
    for subject, records in subject_records.items():
        true = np.concatenate([r["test_true"] for r in records])
        pred = np.concatenate([r["test_pred"] for r in records])
        loss = np.average(
            [r["test_loss"] for r in records],
            weights=[len(r["test_true"]) for r in records],
        )
        subject_metrics[subject] = metrics_for(true, pred, loss)
        subdir = folder / f"sub{subject}"
        save_matrix(subdir / "confusion_matrix_test.png",
                    confusion_matrix(true, pred, labels=range(len(classes))),
                    f"Subject {subject} Test", classes)
        save_matrix(subdir / "confusion_matrix_train.png",
                    sum((r["train_matrix"] for r in records)),
                    f"Subject {subject} Train", classes)

    names = ("accuracy", "macro_f1", "kappa", "balanced_accuracy", "loss")
    overall = {
        name: {
            "mean": float(np.mean([m[name] for m in subject_metrics.values()])),
            "std": float(np.std([m[name] for m in subject_metrics.values()], ddof=0)),
        }
        for name in names
    }
    parameters = {
        "dataset": "BNCI2014-001", "model": MODEL_NAME,
        "data": {**DATA_CONFIG, "data_root": str(DATA_CONFIG["data_root"])},
        "model_config": MODEL_CONFIGS[MODEL_NAME],
        "train": TRAIN_CONFIG, "output_mode": OUTPUT_MODE,
        "classes": classes, "preview": preview,
        "braindecode_version": braindecode.__version__,
        "moabb_version": moabb.__version__,
    }
    with (folder / "metrics.txt").open("w", encoding="utf-8") as file:
        for key, value in parameters.items():
            file.write(f"param.{key}={json.dumps(value, ensure_ascii=False, default=str)}\n")
        for subject, session, fold_id, values in fold_lines:
            file.write(f"fold.{fold_id}.subject={subject}\n")
            file.write(f"fold.{fold_id}.test_session={json.dumps(session)}\n")
            for name, value in values.items():
                file.write(f"fold.{fold_id}.{name}={value:.8f}\n")
            record = recorder.records[fold_id]
            file.write(f"fold.{fold_id}.train_session=\"0train\"\n")
            file.write(f"fold.{fold_id}.n_train={record['n_train']}\n")
            file.write(f"fold.{fold_id}.n_test={record['n_test']}\n")
            file.write(f"fold.{fold_id}.n_validation=0\n")
            for epoch, train_loss in enumerate(record["train_loss"], 1):
                file.write(f"fold.{fold_id}.epoch.{epoch}.train_loss={train_loss:.8f}\n")
        for subject, values in subject_metrics.items():
            for name, value in values.items():
                file.write(f"subject.{subject}.{name}={value:.8f}\n")
        for name, values in overall.items():
            for stat, value in values.items():
                file.write(f"overall.{name}.{stat}={value:.8f}\n")

    report_names = ("accuracy", "macro_f1", "kappa", "loss")
    with (folder / "report.txt").open("w", encoding="utf-8") as file:
        file.write("MOABB + Braindecode EEG 实验报告\n")
        file.write("\n总体（逐受试者均值与标准差）\n")
        for name in report_names:
            file.write(f"  {name + ':':<12}mean: {overall[name]['mean']:.4f},  std: {overall[name]['std']:.4f}\n")
        file.write("\n实验参数\n")
        file.write(json.dumps(parameters, ensure_ascii=False, indent=2, default=str))
        file.write("\n\n划分：每名受试者独立 T(288)→E(288)，验证集 0；E 仅在训练结束后评估。\n")
        file.write("\n逐受试者指标\n")
        for subject, session, fold_id, values in fold_lines:
            file.write(
                f"受试者 {subject}（测试会话 {session}，折 {fold_id}） "
                + " ".join(f"{name}: {values[name]:.4f}" for name in report_names)
                + "\n"
            )
    return subject_metrics, overall

## 7. 官方评估与训练


In [ ]:
def run_experiment():
    # 选择训练对象
    subjects = list(DATA_CONFIG["subjects"])

    if not subjects:
        raise ValueError("subjects 不能为空")
    if MODEL_NAME not in MODEL_CONFIGS:
        raise ValueError(f"未知模型: {MODEL_NAME}")
    if TRAIN_CONFIG["optimizer"] != "Adam" or TRAIN_CONFIG["loss"] != "CrossEntropyLoss":
        raise ValueError("当前仅支持 Adam 和 CrossEntropyLoss")
    if TRAIN_CONFIG["evaluation"] != "train_T_test_E" or TRAIN_CONFIG["validation_split"] is not None:
        raise ValueError("当前固定使用 T→E，validation_split 必须为 None")
    device = TRAIN_CONFIG["device"]
    if device not in ("cpu", "cuda"):
        raise ValueError("device 只能手动设置为 cpu 或 cuda")
    if device == "cuda" and not torch.cuda.is_available():
        raise RuntimeError("已选择 CUDA，但当前环境无法使用 CUDA")
    
    set_random_seeds(seed=TRAIN_CONFIG["seed"], cuda=(device == "cuda"))

    dataset = make_dataset(subjects, DATA_CONFIG["data_root"])
    check_local_data(dataset, subjects)

    paradigm = MotorImagery(
        fmin=None,     # 不做高通滤波
        fmax=None,     # 不做低通滤波
        resample=None, # 不重采样
        tmin=0.0,      # 从事件定义的起点开始
        tmax=None,     # 使用数据集的终点
    )

    preview = load_trial_info(paradigm, dataset, subjects[0])
    n_trials, n_chans, n_times = preview["shape"]
    n_classes = len(preview["classes"])
    print(f"受试者: {subjects}；设备: {device}")
    print(f"首名受试者 Trial: {preview['shape']}；采样率: {preview['sfreq']} Hz")
    print(f"类别分布: {preview['counts']}；会话: {preview['sessions']}")
    print("每名受试者独立 T(288)→E(288)，训练内部不再划分；E 仅在训练结束后评估")
    print(f"预期输入: ({n_trials},1,{n_chans},{n_times})；类别数: {n_classes}")
    print("\n")

    model = build_model(MODEL_NAME, n_chans, n_times, n_classes)
    check_model(model, MODEL_NAME, n_chans, n_times, n_classes)
    model.zero_grad(set_to_none=True)
    
    # 这里将自定义的 recorder 注册为准确率评分函数。
    recorder = FoldScorer(n_classes, SAVE_CHECKPOINT)
    paradigm.scorer = {"accuracy": recorder, "fold_id": recorder.fold_id}

    # 配置神经网络训练器
    classifier = ReportingEEGClassifier(
        module=model,
        criterion=nn.CrossEntropyLoss,
        optimizer=torch.optim.Adam,
        optimizer__lr=TRAIN_CONFIG["learning_rate"],
        optimizer__weight_decay=TRAIN_CONFIG["weight_decay"],
        max_epochs=TRAIN_CONFIG["epochs"],
        batch_size=TRAIN_CONFIG["batch_size"],
        train_split=None,
        classes=np.arange(n_classes),
        device=device,
        iterator_train__drop_last=False,
        callbacks=[("best_train_loss", BestTrainLoss()),],
    )

    """
    数据必须先经过 adapt_input()，然后才能进入模型。
    例如：(B,22,1001)->adapt->(B,1,22,1001)->EEGTCNet->(B,4)
    这正是 data_adapter.py 与 eegnet_model.py 的连接位置。
    """
    pipeline = Pipeline([
        (
            "standardize",ChannelStandardizer()
        ),
        # "adapt" 和 "classifier" 是为步骤指定的名称
        (
            "adapt", 
            # FunctionTransformer：将一个普通的 Python 函数包装为 Scikit-learn 可以使用的 Transformer。
            FunctionTransformer(
                adapt_input, 
                kw_args={"model_name": MODEL_NAME}, 
                validate=False)
        ),
        (
            "classifier", classifier
        ),]
    )

    folder = output_dir()
    folder.mkdir(parents=True, exist_ok=True)
    run_id = datetime.now().strftime("%Y%m%d_%H%M%S_%f")

    # 启动训练和评估 启动完整的读取数据、划分训练测试集、训练模型、测试模型和记录结果的流程
    # CrossSessionEvaluation 是 MOABB 的跨 Session 评估器。
    common = dict(
        paradigm=paradigm, datasets=[dataset], random_state=TRAIN_CONFIG["seed"],
        n_jobs=1, hdf5_path=str(folder), suffix=run_id,
        cache_config={"save_raw": False, "save_epochs": False, "save_array": False, "use": False},
    )
    evaluator = CrossSessionEvaluation(
        **common, cv_class=TTrainETestSplit
    )
    print(f"评估: {TRAIN_CONFIG['evaluation']}；结果目录: {folder}")
    results = evaluator.process({MODEL_NAME: pipeline})

    if len(results) != len(subjects) or len(recorder.records) != len(subjects):
        raise RuntimeError("每名受试者必须只有一次 T→E 训练与测试")
    if set(results["subject"].astype(int)) != set(subjects) or not (results["session"] == "1test").all():
        raise RuntimeError("评估结果的受试者或测试会话不符合 T→E 协议")
    subject_metrics, overall = write_results(
        folder, results, recorder, preview["classes"], preview
    )
    print("逐受试者指标:", subject_metrics)
    print("总体均值与标准差:", overall)
    return results, subject_metrics, overall

## 8. 执行实验


In [8]:
# 运行当前选择的一个模型；默认 9 名受试者、500 Epoch
results, subject_metrics, overall = run_experiment()

The dataset class name 'LocalBNCI2014-001' must be an abbreviation of its code 'BNCI2014-001'. See moabb.datasets.base.is_abbrev for more information.
Choosing from all possible events
c:\Users\hp\.conda\envs\moabb\Lib\site-packages\moabb\datasets\bnci\base.py:456: FutureWarning: Montage name 'standard_1005' is deprecated and will be removed in MNE 1.14. Use 'colin27_1005' instead.
  montage = make_standard_montage("standard_1005")
c:\Users\hp\.conda\envs\moabb\Lib\site-packages\moabb\datasets\bnci\base.py:456: FutureWarning: Montage name 'standard_1005' is deprecated and will be removed in MNE 1.14. Use 'colin27_1005' instead.
  montage = make_standard_montage("standard_1005")
c:\Users\hp\.conda\envs\moabb\Lib\site-packages\moabb\datasets\bnci\base.py:456: FutureWarning: Montage name 'standard_1005' is deprecated and will be removed in MNE 1.14. Use 'colin27_1005' instead.
  montage = make_standard_montage("standard_1005")
c:\Users\hp\.conda\envs\moabb\Lib\site-packages\moabb\datasets

Adding metadata with 3 columns
Adding metadata with 3 columns
Adding metadata with 3 columns
Adding metadata with 3 columns
Adding metadata with 3 columns
Adding metadata with 3 columns
Adding metadata with 3 columns
Adding metadata with 3 columns
Adding metadata with 3 columns
Adding metadata with 3 columns
Adding metadata with 3 columns
Adding metadata with 3 columns
Adding metadata with 3 columns
576 matching events found
No baseline correction applied
受试者: [1, 2, 3, 4, 5, 6, 7, 8, 9]；设备: cuda
首名受试者 Trial: (576, 22, 1001)；采样率: 250.0 Hz
类别分布: {'feet': 144, 'left_hand': 144, 'right_hand': 144, 'tongue': 144}；会话: {'0train': 288, '1test': 288}
每名受试者独立 T(288)→E(288)，训练内部不再划分；E 仅在训练结束后评估
预期输入: (576,1,22,1001)；类别数: 4


模型输入: (32, 1, 22, 1001) -> 实际输入: (32, 1, 22, 1001)
模型输出: (32, 4)；可训练参数: 70,916
CrossEntropyLoss 前向/反向检查通过: 1.3884


评估: train_T_test_E；结果目录: D:\Data\Code\Python\脑机\BCI\results\EEGTCNet


c:\Users\hp\.conda\envs\moabb\Lib\site-packages\torch\nn\modules\conv.py:548: UserWarning: Using padding='same' with even kernel lengths and odd dilation may require a zero-padded copy of the input be created (Triggered internally at C:\actions-runner\_work\pytorch\pytorch\pytorch\aten\src\ATen\native\Convolution.cpp:1025.)
  return F.conv2d(
c:\Users\hp\.conda\envs\moabb\Lib\site-packages\moabb\datasets\bnci\base.py:456: FutureWarning: Montage name 'standard_1005' is deprecated and will be removed in MNE 1.14. Use 'colin27_1005' instead.
  montage = make_standard_montage("standard_1005")
c:\Users\hp\.conda\envs\moabb\Lib\site-packages\moabb\datasets\bnci\base.py:456: FutureWarning: Montage name 'standard_1005' is deprecated and will be removed in MNE 1.14. Use 'colin27_1005' instead.
  montage = make_standard_montage("standard_1005")
c:\Users\hp\.conda\envs\moabb\Lib\site-packages\moabb\datasets\bnci\base.py:456: FutureWarning: Montage name 'standard_1005' is deprecated and will be re

T→E 划分：训练 288，测试 288，验证 0
T→E 划分：训练 288，测试 288，验证 0
T→E 划分：训练 288，测试 288，验证 0
T→E 划分：训练 288，测试 288，验证 0
T→E 划分：训练 288，测试 288，验证 0
T→E 划分：训练 288，测试 288，验证 0
T→E 划分：训练 288，测试 288，验证 0
T→E 划分：训练 288，测试 288，验证 0
T→E 划分：训练 288，测试 288，验证 0
  epoch    train_loss     dur
-------  ------------  ------
      1        1.4655  0.3988
      2        1.3739  0.0730
      3        1.3572  0.0733
      4        1.3293  0.0711
      5        1.2537  0.0700
      6        1.2132  0.0720
      7        1.1722  0.0750
      8        1.1366  0.0760
      9        1.0979  0.0730
     10        1.0718  0.0710
     11        0.9780  0.0710
     12        0.9696  0.0712
     13        0.8571  0.0745
     14        0.8002  0.0739
     15        0.7917  0.0750
     16        0.7535  0.0740
     17        0.7671  0.0730
     18        0.6708  0.0778
     19        0.6574  0.0690
     20        0.6667  0.0700
     21        0.5927  0.0700
     22        0.6007  0.0690
     23        0.6167  0.0680
     24        0

c:\Users\hp\.conda\envs\moabb\Lib\site-packages\moabb\analysis\results.py:189: H5pyDeprecationWarning: Creating a dataset without passing data or dtype is deprecated. Pass an explicit dtype. Using dtype='f4' will keep the current default behaviour.
  dset.create_dataset(


逐受试者指标: {1: {'accuracy': 0.8229166666666666, 'macro_f1': 0.8208499339285178, 'kappa': 0.7638888888888888, 'balanced_accuracy': 0.8229166666666666, 'loss': np.float64(0.5510136485099792)}, 2: {'accuracy': 0.53125, 'macro_f1': 0.5329237112619802, 'kappa': 0.375, 'balanced_accuracy': 0.53125, 'loss': np.float64(1.5449552536010742)}, 3: {'accuracy': 0.8923611111111112, 'macro_f1': 0.8917924462196062, 'kappa': 0.8564814814814815, 'balanced_accuracy': 0.8923611111111112, 'loss': np.float64(0.38660115003585815)}, 4: {'accuracy': 0.7118055555555556, 'macro_f1': 0.7110786809149172, 'kappa': 0.6157407407407407, 'balanced_accuracy': 0.7118055555555556, 'loss': np.float64(0.954712450504303)}, 5: {'accuracy': 0.7777777777777778, 'macro_f1': 0.7726226275075201, 'kappa': 0.7037037037037037, 'balanced_accuracy': 0.7777777777777778, 'loss': np.float64(0.7336318492889404)}, 6: {'accuracy': 0.6284722222222222, 'macro_f1': 0.6278809098339083, 'kappa': 0.5046296296296297, 'balanced_accuracy': 0.62847222222

## 9. 多模型历史结果对比（待后续优化）


In [9]:
def compare_models_from_txt(*args, **kwargs):
    # 待用户后续优化：读取历史 TXT 并比较多个模型；本阶段不实现。
    pass